<h1> Best Selling Item </h1>

In [ ]:
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, FloatType, IntegerType

# Define schema
schema = StructType([
    StructField("invoicedate", TimestampType(), True),
    StructField("description", StringType(), True),
    StructField("unitprice", FloatType(), True),
    StructField("quantity", IntegerType(), True)
])

# Define data
data = [
    ("2023-01-15 10:00:00", "Item A", 10.00, 2),
    ("2023-01-15 11:00:00", "Item B", 20.00, 1),
    ("2023-02-20 12:00:00", "Item A", 10.00, 5),
    ("2023-02-25 13:00:00", "Item C", 15.00, 3),
    ("2023-03-05 14:00:00", "Item B", 20.00, 4),
    ("2023-03-10 15:00:00", "Item C", 15.00, 2),
    ("2023-03-15 16:00:00", "Item A", 10.00, 1),
    ("2023-04-01 17:00:00", "Item D", 25.00, 2),
    ("2023-04-05 18:00:00", "Item B", 20.00, 3),
    ("2023-04-10 19:00:00", "Item C", 15.00, 4),
    ("2023-05-15 20:00:00", "Item A", 10.00, 6)
]

# Create DataFrame
online_retail = spark.createDataFrame(data, schema)

In [ ]:
#import necessary functions
import pyspark
from pyspark.sql.functions import col,month,sum,row_number
from pyspark.sql.window import Window

In [ ]:
# Start writing code
online_retail_df=online_retail.selectExpr('(unitprice*quantity) as total_paid','month(invoicedate) as month','description').\
groupBy('month','description').\
agg(sum(col('total_paid')).alias('total_paid')).\
select("month","description","total_paid")

In [ ]:
window_spec=Window.partitionBy(col('month')).orderBy(col('total_paid').desc())
online_retail_df=online_retail_df.withColumn("rn",row_number().over(window_spec)).filter("rn=1").drop("rn").orderBy(col("month").asc_nulls_last())